In [1]:
import pandas as pd
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import joblib

# ===================== 配置路径（适配你的桌面） =====================
BASE_PATH = "C:\\Users\\sdliu\\Desktop\\文献\\"

# ===================== 步骤1：加载划分好的数据集 =====================
print("正在加载数据集...")
train_df = pd.read_csv(BASE_PATH + "train_data.csv")
val_df = pd.read_csv(BASE_PATH + "val_data.csv")
test_df = pd.read_csv(BASE_PATH + "test_data.csv")

# 检查数据是否加载成功
print(f"训练集样本数: {len(train_df)}, 验证集: {len(val_df)}, 测试集: {len(test_df)}")

# ===================== 步骤2：定义文本预处理函数 =====================
def preprocess_text(text):
    """
    对NOTAM文本进行预处理：
    1. 转为小写
    2. 中文分词（使用jieba）
    3. 去除停用词和无意义字符（这里简化处理，可根据需要扩展）
    """
    # 转为小写（针对英文部分）
    text = str(text).lower()
    
    # 中文分词
    words = jieba.lcut(text)
    
    # 过滤掉长度小于1的词（去除标点、空格等）
    words = [word for word in words if len(word) > 1]
    
    # 返回用空格连接的分词结果
    return " ".join(words)

# ===================== 步骤3：对所有文本进行预处理 =====================
print("正在进行文本预处理与分词...")
train_df["processed_text"] = train_df["e_text"].apply(preprocess_text)
val_df["processed_text"] = val_df["e_text"].apply(preprocess_text)
test_df["processed_text"] = test_df["e_text"].apply(preprocess_text)

# 查看预处理后的样本
print("\n预处理后的文本样本（前3条）：")
print(train_df[["e_text", "processed_text"]].head(3))

# ===================== 步骤4：使用TF-IDF提取文本特征 =====================
print("\n正在提取TF-IDF特征...")

# 初始化TF-IDF向量化器
# 这里设置max_features=5000，表示只保留最重要的5000个词作为特征
# 你可以根据实际情况调整这个数值
tfidf = TfidfVectorizer(max_features=2000)

# 在训练集上拟合（学习词汇表），并转换为特征矩阵
X_train = tfidf.fit_transform(train_df["processed_text"])

# 在验证集和测试集上直接转换（使用训练集的词汇表）
X_val = tfidf.transform(val_df["processed_text"])
X_test = tfidf.transform(test_df["processed_text"])

print(f"TF-IDF特征矩阵维度：")
print(f"训练集: {X_train.shape}, 验证集: {X_val.shape}, 测试集: {X_test.shape}")

# ===================== 步骤5：对标签q_code进行数值编码 =====================
print("\n正在对标签进行编码...")

# 初始化标签编码器
label_encoder = LabelEncoder()

# 在训练集上拟合（学习所有q_code类别），并转换为数值
y_train = label_encoder.fit_transform(train_df["q_code"])

# 在验证集和测试集上转换
y_val = label_encoder.transform(val_df["q_code"])
y_test = label_encoder.transform(test_df["q_code"])

# 查看标签映射关系
print(f"标签类别数量: {len(label_encoder.classes_)}")
print(f"前5个标签及其编码: {dict(zip(label_encoder.classes_[:5], label_encoder.transform(label_encoder.classes_[:5])))}")

# ===================== 步骤6：保存预处理后的特征和模型 =====================
print("\n正在保存预处理结果...")

# 保存TF-IDF向量化器和标签编码器（后续预测时需要用到）
joblib.dump(tfidf, BASE_PATH + "tfidf_vectorizer.pkl")
joblib.dump(label_encoder, BASE_PATH + "label_encoder.pkl")

# 保存特征矩阵和标签（为了方便，我们也可以保存为稀疏矩阵格式）
joblib.dump((X_train, y_train), BASE_PATH + "train_features.pkl")
joblib.dump((X_val, y_val), BASE_PATH + "val_features.pkl")
joblib.dump((X_test, y_test), BASE_PATH + "test_features.pkl")

print("\n✅ 第二步骤完成！所有文件已保存至：")
print(BASE_PATH)
print("\n生成的文件清单：")
print("- tfidf_vectorizer.pkl (TF-IDF向量化器)")
print("- label_encoder.pkl (标签编码器)")
print("- train_features.pkl (训练集特征与标签)")
print("- val_features.pkl (验证集特征与标签)")
print("- test_features.pkl (测试集特征与标签)")

Building prefix dict from the default dictionary ...


正在加载数据集...
训练集样本数: 10591, 验证集: 1324, 测试集: 1324
正在进行文本预处理与分词...


Dumping model to file cache C:\Users\sdliu\AppData\Local\Temp\jieba.cache
Loading model cost 0.467 seconds.
Prefix dict has been built successfully.



预处理后的文本样本（前3条）：
                                              e_text  \
0                             RWY36ILS仅供测试,不可使用,因校飞.   
1  RWY17L/35R关闭(允许从P1,P2,P5,P6滑行道穿越RWY17L/35R)跑道关...   
2  关闭下列滑行道:1.Z4滑行道.2.Z5滑行道.3.T4滑行道以北的Z1滑行道.4.T4滑行...   

                                      processed_text  
0                            rwy36ils 仅供 测试 不可 使用 因校  
1  rwy17l 35r 关闭 允许 p1 p2 p5 p6 滑行道 穿越 rwy17l 35r...  
2  关闭 下列 滑行道 z4 滑行道 z5 滑行道 t4 滑行道 以北 z1 滑行道 t4 滑行...  

正在提取TF-IDF特征...
TF-IDF特征矩阵维度：
训练集: (10591, 2000), 验证集: (1324, 2000), 测试集: (1324, 2000)

正在对标签进行编码...
标签类别数量: 13
前5个标签及其编码: {'A': 0, 'F': 1, 'I': 2, 'K': 3, 'L': 4}

正在保存预处理结果...

✅ 第二步骤完成！所有文件已保存至：
C:\Users\sdliu\Desktop\文献\

生成的文件清单：
- tfidf_vectorizer.pkl (TF-IDF向量化器)
- label_encoder.pkl (标签编码器)
- train_features.pkl (训练集特征与标签)
- val_features.pkl (验证集特征与标签)
- test_features.pkl (测试集特征与标签)


In [2]:
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ===================== 1. 配置路径（适配你的桌面） =====================
BASE_PATH = "C:\\Users\\sdliu\\Desktop\\文献\\"

# ===================== 2. 加载预处理好的特征 =====================
print("正在加载特征数据...")
X_train, y_train = joblib.load(BASE_PATH + "train_features.pkl")
X_val, y_val = joblib.load(BASE_PATH + "val_features.pkl")
X_test, y_test = joblib.load(BASE_PATH + "test_features.pkl")

# 加载标签编码器（用于把数值转回Q码）
label_encoder = joblib.load(BASE_PATH + "label_encoder.pkl")

print(f"数据加载完成！训练集样本数: {X_train.shape[0]}")

# ===================== 3. 训练模型（逻辑回归，快速基线） =====================
print("\n正在训练逻辑回归模型...")
# 初始化模型（max_iter设大一点确保收敛）
model = LogisticRegression(max_iter=1000, random_state=42)

# 开始训练
model.fit(X_train, y_train)
print("模型训练完成！")

# ===================== 4. 在验证集上评估效果 =====================
print("\n正在验证集上评估...")
y_val_pred = model.predict(X_val)

# 计算准确率
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"✅ 验证集准确率: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

# 打印详细分类报告（包含每个Q码的精确率、召回率）
print("\n详细分类报告：")
print(classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))

# ===================== 5. 在测试集上最终测试（可选） =====================
print("\n正在测试集上最终测试...")
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"✅ 测试集最终准确率: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# ===================== 6. 保存训练好的模型 =====================
joblib.dump(model, BASE_PATH + "qcode_classifier_model.pkl")
print(f"\n✅ 模型已保存至：{BASE_PATH}qcode_classifier_model.pkl")

正在加载特征数据...
数据加载完成！训练集样本数: 10591

正在训练逻辑回归模型...
模型训练完成！

正在验证集上评估...
✅ 验证集准确率: 0.9539 (95.39%)

详细分类报告：
              precision    recall  f1-score   support

           A       0.96      0.99      0.97        90
           F       0.96      0.89      0.92        90
           I       0.94      0.92      0.93       255
           K       1.00      1.00      1.00         4
           L       0.67      0.20      0.31        10
           M       0.95      1.00      0.97       717
           N       1.00      0.90      0.95        70
           O       1.00      0.94      0.97        18
           P       1.00      0.89      0.94        19
           R       1.00      0.90      0.95        10
           S       1.00      0.67      0.80         3
           W       1.00      1.00      1.00        11
           X       1.00      0.70      0.83        27

    accuracy                           0.95      1324
   macro avg       0.96      0.85      0.89      1324
weighted avg       0.95      0

In [9]:
import jieba
import joblib

# ===================== 配置路径 =====================
BASE_PATH = "C:\\Users\\sdliu\\Desktop\\文献\\"

# ===================== 加载模型和编码器 =====================
model = joblib.load(BASE_PATH + "qcode_classifier_model.pkl")
tfidf = joblib.load(BASE_PATH + "tfidf_vectorizer.pkl")
label_encoder = joblib.load(BASE_PATH + "label_encoder.pkl")

# ===================== 文本预处理（和训练一致） =====================
def preprocess_text(text):
    text = str(text).lower()
    words = jieba.lcut(text)
    words = [word for word in words if len(word) > 1]
    return " ".join(words)

# ===================== 预测函数 =====================
def predict_qcode(e_text):
    processed = preprocess_text(e_text)
    features = tfidf.transform([processed])
    pred_num = model.predict(features)[0]
    q_code = label_encoder.inverse_transform([pred_num])[0]
    return q_code

# ===================== 【核心】终端输入 =====================
if __name__ == "__main__":
    print("="*60)
    print("📝 请在下方输入/粘贴 E项文本，输入完成后按回车")
    print("="*60)
    
    # 终端获取输入（支持粘贴超长文本）
    input_e_text = input("请输入E项文本：")
    
    # 执行预测
    result = predict_qcode(input_e_text)
    
    # 输出结果
    print("\n" + "="*60)
    print(f"✅ 预测完成！对应的 Q代码 为：【{result}】")
    print("="*60)

📝 请在下方输入/粘贴 E项文本，输入完成后按回车



✅ 预测完成！对应的 Q代码 为：【M】
